#### Keyword-search

Explore the MITRE D3FEND knowledge base with SPARQL (with rdflib).

Key structure of the ontology (useful to read the queries):
  * A defensive technique is an OWL *class* under rdfs:subClassOf* d3f:DefensiveTechnique,
    carrying rdfs:label, d3f:d3fend-id ("D3-xx") and d3f:definition.
  * Its TACTIC is given by an :enables restriction on an ancestor:
        ?tech rdfs:subClassOf [ owl:onProperty d3f:enables ;
                                owl:someValuesFrom <Tactic> ]
  * Its DIGITAL ARTIFACTS are given by other restrictions, where the verb
    is the property (:disables, :restricts, :analyzes, ...):
        ?tech rdfs:subClassOf [ owl:onProperty <verb> ;
                                owl:someValuesFrom <Artifact> ]
  * rdfs:subClassOf between two techniques = sub-technique relation.
  * d3f:synonym gives alternative names.

In [93]:
from rdflib import Graph,Literal,URIRef

In [94]:
g = Graph().parse("d3fend.ttl", format="turtle")

In [95]:
P = """
PREFIX d3f:  <http://d3fend.mitre.org/ontologies/d3fend.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
"""

In [96]:
def run(title, q, limit=8):
    print(f"\n===== {title} =====")
    rows = list(g.query(P + q))
    for r in rows[:limit]:
        print("  " + " | ".join(
            "" if v is None else str(v).split("#")[-1] for v in r))
    print(f"  ({len(rows)} rows)")

OPERATIONALIZATION PIPELINE

Example of leaf goal: "Remove access rights of employees"


Chain:
1) Find artifact
2) techniques on artifact
3) filter by verb
4) verify

For another goal, change: the keyword(s), the artifact, the verbs, the ids.

In [97]:
# STEP 1: Find the digital artifact the goal concerns.
# Keyword-search the ARTIFACT taxonomy with a noun from the goal's object
# ("access rights of employees" --> try 'account', then 'access', 'credential').
# Run it once per candidate term and keep the union; pick the artifact that
# best matches the goal's object (here: User Account).
run("STEP 1  find artifact (keyword 'account')", """
SELECT ?label WHERE {
  ?a rdfs:subClassOf* d3f:DigitalArtifact ; rdfs:label ?label .
  FILTER( CONTAINS(LCASE(?label), "account") )
} ORDER BY ?label""")


===== STEP 1  find artifact (keyword 'account') =====
  Cloud User Account
  Default User Account
  Domain User Account
  Global User Account
  Kerberos Ticket Granting Ticket Account
  Local User Account
  Privileged User Account
  Service Account
  (9 rows)


In [98]:
# STEP 1b (optional): Confirm the chosen artifact by reading its definition,
# to be sure it really denotes the goal's object before going further.
run("STEP 1b  confirm artifact (UserAccount)", """
SELECT ?definition WHERE {
  d3f:UserAccount d3f:definition ?definition .
}""")


===== STEP 1b  confirm artifact (UserAccount) =====
  A user account allows a user to authenticate to a system and potentially to receive authorization to access resources provided by or connected to that system; however, authentication does not imply authorization. To log into an account, a user is typically required to authenticate oneself with a password or other credentials for the purposes of accounting, security, logging, and resource management.
  (1 rows)


In [99]:
# STEP 2: All techniques operating on that artifact.
# The restriction [ owl:onProperty ?verb ; owl:someValuesFrom <Artifact> ]
# is how D3FEND links a technique to the artifact it acts on; ?verb tells you
# HOW it acts (disables, restricts, strengthens, restores, maps...).
run("STEP 2  techniques operating on UserAccount", """
SELECT ?id ?label ?verb WHERE {
  ?t rdfs:subClassOf* d3f:DefensiveTechnique ;
     d3f:d3fend-id ?id ; rdfs:label ?label .
  ?t rdfs:subClassOf [ owl:onProperty ?verb ;
                       owl:someValuesFrom d3f:UserAccount ] .
} ORDER BY ?id""")


===== STEP 2  techniques operating on UserAccount =====
  D3-AA | Agent Authentication | strengthens
  D3-AL | Account Locking | disables
  D3-AM | Access Modeling | maps
  D3-CDP | Change Default Password | strengthens
  D3-RUAA | Restore User Account Access | restores
  D3-UAP | User Account Permissions | restricts
  D3-ULA | Unlock Account | restores
  (7 rows)


In [100]:
# STEP 3: Filter by the verb matching the goal's intent.
#   "Remove" --> keep disables/restricts (Evict/Harden); drop strengthens,
#   restores, maps (wrong or opposite intent). Just add the FILTER to STEP 2.
run("STEP 3  filter by verb (disables/restricts)", """
SELECT ?id ?label ?verb WHERE {
  ?t rdfs:subClassOf* d3f:DefensiveTechnique ;
     d3f:d3fend-id ?id ; rdfs:label ?label .
  ?t rdfs:subClassOf [ owl:onProperty ?verb ;
                       owl:someValuesFrom d3f:UserAccount ] .
  FILTER( ?verb IN (d3f:disables, d3f:restricts) )
} ORDER BY ?id""")


===== STEP 3  filter by verb (disables/restricts) =====
  D3-AL | Account Locking | disables
  D3-UAP | User Account Permissions | restricts
  (2 rows)


In [101]:
# STEP 4: Verify the retained techniques by their definition before attaching them as actions.
# List every retained id at once with IN(...).
run("STEP 4  verify definitions of retained techniques", """
SELECT ?id ?label ?definition WHERE {
  ?t d3f:d3fend-id ?id ; rdfs:label ?label ; d3f:definition ?definition .
  FILTER( ?id IN ("D3-AL", "D3-UAP") )
} ORDER BY ?id""")


===== STEP 4  verify definitions of retained techniques =====
  D3-AL | Account Locking | The process of temporarily disabling user accounts on a system or domain.
  D3-UAP | User Account Permissions | Restricting a user account's access to resources.
  (2 rows)


Leaf goal : Remove access rights of employees

Actions :
- D3-AL Account Locking
- D3-UAP User Account Permissions

-------

ALL DEFENSIVE TECHNIQUES

In [102]:
results = """
PREFIX d3f:  <http://d3fend.mitre.org/ontologies/d3fend.owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?technique ?id ?label ?definition WHERE {
    ?technique rdfs:subClassOf* d3f:DefensiveTechnique ;
               d3f:d3fend-id ?id ;
               rdfs:label ?label .

    OPTIONAL { ?technique d3f:definition ?definition . }
}"""

technique_rows = list(g.query(results))


for i, row in enumerate(sorted(technique_rows, key=lambda r: (str(r.id), str(r.label))), 1):
    uri = str(row.technique)
    d3fend_id = str(row.id).strip()
    label = str(row.label).strip()
    definition = str(row.definition).strip() if row.definition else ""

    print(f"{i:3d}. {d3fend_id:12s} {label}")
    if definition:
        print(f"     Definition : {definition}")

print(f"\nTotal: {len(technique_rows)}")

  1. D3-AA        Agent Authentication
     Definition : Agent authentication is the process of verifying the identities of agents to ensure they are authorized and trustworthy participants within a system.
  2. D3-ABPI      Application-based Process Isolation
     Definition : Application code which prevents its own subroutines from accessing intra-process / internal memory space.
  3. D3-ACA       Active Certificate Analysis
     Definition : Actively collecting PKI certificates by connecting to the server and downloading its server certificates for analysis.
  4. D3-ACH       Application Configuration Hardening
     Definition : Modifying an application's configuration to reduce its attack surface.
  5. D3-AEM       Application Exception Monitoring
     Definition : Monitoring the failures of system counters and timers.
  6. D3-AH        Application Hardening
     Definition : Application Hardening makes an executable application more resilient to a class of exploits which either in